# 🎗️ Breast Cancer Classification
**Dataset:** Wisconsin Breast Cancer Dataset — 569 samples, 30 features  
**Goal:** Classify tumours as Malignant (1) or Benign (0) using Logistic Regression and MLP Neural Network  
**Author:** Arman Arabkhani | AUT Data Science


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, roc_curve, auc)


## 2. Load & Explore Data

In [ ]:
BC_df = pd.read_csv('Breast_cancer.csv')
print("Shape:", BC_df.shape)
BC_df.head()


In [ ]:
# Missing values
print("Missing values:\n", BC_df.isnull().sum().sum(), "total")

# Summary statistics
BC_df.describe()


In [ ]:
# Class distribution
ax = sns.countplot(x='diagnosis', data=BC_df, palette='flare')
ax.set_title('Class Distribution (M = Malignant, B = Benign)', fontsize=14)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + 0.35, p.get_height() - 20),
                ha='center', color='white', fontsize=13)
plt.tight_layout()
plt.show()

print(BC_df['diagnosis'].value_counts())
print("\nNote: Dataset is slightly imbalanced — 357 Benign vs 212 Malignant")


## 3. Preprocessing
**Steps:**
- Encode target variable (M → 1, B → 0)
- Stratified train/test split (70/30) — preserves class ratio
- Fit StandardScaler **only on training data** to prevent data leakage


In [ ]:
# Features and target
X = BC_df.iloc[:, 1:31].values
y = BC_df.iloc[:, 31].values

# Encode target
le = LabelEncoder()
y = le.fit_transform(y)

# ✅ Split BEFORE scaling to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=3
)

# ✅ Fit scaler on training data only, then transform both sets
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)   # transform only — never fit on test data

print(f"Training set: {X_train.shape} | Test set: {X_test.shape}")


## 4. Model 1 — Logistic Regression (Baseline)

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=3)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

print(f"Logistic Regression — Test Accuracy: {accuracy_score(y_test, lr_pred)*100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_test, lr_pred, target_names=['Benign', 'Malignant']))


In [ ]:
# Confusion matrix — Logistic Regression
outcome_labels = ['Benign', 'Malignant']
plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, lr_pred), annot=True, fmt='d',
            cmap='Blues', xticklabels=outcome_labels, yticklabels=outcome_labels)
plt.title('Confusion Matrix — Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()


## 5. Model 2 — MLP Neural Network

In [ ]:
mlp = MLPClassifier(max_iter=500, alpha=0.1, activation='logistic',
                    solver='adam', random_state=3, verbose=False)
mlp.fit(X_train, y_train)
mlp_pred = mlp.predict(X_test)

print(f"MLP — Training Accuracy:  {mlp.score(X_train, y_train)*100:.2f}%")
print(f"MLP — Test Accuracy:      {accuracy_score(y_test, mlp_pred)*100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_test, mlp_pred, target_names=['Benign', 'Malignant']))


In [ ]:
# Loss curve
plt.figure(figsize=(7, 4))
plt.plot(mlp.loss_curve_)
plt.title('MLP Training Loss Curve')
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Confusion matrix — MLP
plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, mlp_pred), annot=True, fmt='d',
            cmap='Oranges', xticklabels=outcome_labels, yticklabels=outcome_labels)
plt.title('Confusion Matrix — MLP')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()


## 6. ROC Curves — Both Models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, model, pred, name, color in zip(
        axes,
        [lr, mlp],
        [lr_pred, mlp_pred],
        ['Logistic Regression', 'MLP'],
        ['steelblue', 'darkorange']):

    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)

    ax.plot(fpr, tpr, color=color, lw=2, label=f'AUC = {roc_auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve — {name}')
    ax.legend(loc='lower right')

plt.tight_layout()
plt.show()


## 7. Cross-Validation (10-Fold Stratified)
Cross-validation gives a more reliable estimate of performance by testing across all data splits.


In [ ]:
# Re-combine scaled data for cross-validation
X_all = np.vstack([X_train, X_test])
y_all = np.concatenate([y_train, y_test])

kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)

# Cross-validate both models
lr_cv_scores  = cross_val_score(lr,  X_all, y_all, cv=kfold, scoring='accuracy')
mlp_cv_scores = cross_val_score(mlp, X_all, y_all, cv=kfold, scoring='accuracy')

print(f"Logistic Regression  — CV Accuracy: {lr_cv_scores.mean()*100:.2f}% ± {lr_cv_scores.std()*100:.2f}%")
print(f"MLP Neural Network   — CV Accuracy: {mlp_cv_scores.mean()*100:.2f}% ± {mlp_cv_scores.std()*100:.2f}%")


In [ ]:
# K-Fold confusion matrix for MLP
predicted_targets = np.array([])
actual_targets    = np.array([])

for train_ix, test_ix in kfold.split(X_all, y_all):
    X_tr, X_te = X_all[train_ix], X_all[test_ix]
    y_tr, y_te = y_all[train_ix], y_all[test_ix]
    mlp.fit(X_tr, y_tr)
    predicted_targets = np.append(predicted_targets, mlp.predict(X_te))
    actual_targets    = np.append(actual_targets,    y_te)

print(f"MLP K-Fold Overall Accuracy: {accuracy_score(actual_targets, predicted_targets)*100:.2f}%")

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(actual_targets, predicted_targets).astype(int),
            annot=True, fmt='d', cmap='Blues',
            xticklabels=outcome_labels, yticklabels=outcome_labels)
plt.title('MLP Confusion Matrix — 10-Fold CV')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()


## 8. Model Comparison

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

results = {
    'Model':     ['Logistic Regression', 'MLP Neural Network'],
    'Accuracy':  [accuracy_score(y_test, lr_pred),  accuracy_score(y_test, mlp_pred)],
    'Precision': [precision_score(y_test, lr_pred), precision_score(y_test, mlp_pred)],
    'Recall':    [recall_score(y_test, lr_pred),    recall_score(y_test, mlp_pred)],
    'F1-Score':  [f1_score(y_test, lr_pred),        f1_score(y_test, mlp_pred)],
}
results_df = pd.DataFrame(results).set_index('Model')
print(results_df.round(4))

# Bar chart
results_df.plot(kind='bar', figsize=(8, 5))
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.ylim(0.9, 1.01)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


## 9. Conclusion

- **MLP outperformed Logistic Regression** overall, capturing non-linear relationships in the data.
- **Recall for Malignant class** is the most critical metric in this context — a false negative means missing a cancer diagnosis, which is far more dangerous than a false positive.
- **Data leakage was avoided** by fitting the StandardScaler on training data only before applying it to the test set.
- Cross-validation confirmed consistent performance across all splits with low variance.

### 🔧 Future Improvements
- Tune MLP architecture (hidden layers, neurons) using GridSearchCV
- Test additional models: Random Forest, SVM, XGBoost
- Address class imbalance using SMOTE or class-weight adjustment
- Prioritise recall for the malignant class using a threshold-tuning approach
